# Cell Segmentation, From Tensor to Service

This notebook is the **teaching artifact** of the project. It walks through every concept you need to be able to explain in an interview, in the order you should learn them.

Read every markdown cell. Run every code cell. When something looks like magic, change a number and run it again to see what happens. You will not learn this by reading alone.

## What this notebook covers

1. What image segmentation actually is, and why it matters in life sciences
2. How PyTorch represents an image (tensors, shapes, dtypes)
3. What Cellpose is, in one paragraph
4. Loading the model and running it on a single image
5. Inspecting the raw outputs (masks, cell IDs, flows)
6. Visualizing the result
7. Measuring CPU vs GPU latency (the deployment-engineering bit)
8. Extracting cell counts and basic morphology
9. How this notebook becomes the FastAPI service in `backend/`

## Before you run anything

Activate the venv and select its kernel in VS Code or Jupyter:

```
C:\Users\palla\venvs\sartorius-cell
```

If you see import errors, the kernel is wrong — check the kernel picker in the top right.

## 1. What is image segmentation?

Three related tasks in computer vision, easy to mix up:

| Task | Output | Example |
|---|---|---|
| **Classification** | One label for the whole image | "This image contains cells" |
| **Object detection** | Bounding boxes around things | A rectangle around each cell |
| **Segmentation** | A label for every pixel | This pixel belongs to cell #5 |

Within segmentation there are two flavors:

- **Semantic segmentation** — every pixel labeled with a class (cell vs background). All cells share one label.
- **Instance segmentation** — every pixel labeled with a class AND an instance ID. Cell #1, cell #2, cell #3 are distinct.

Cellpose does **instance segmentation**. That is what Sartorius's Kaggle competition was about and what their imaging products need: not just "there are cells here" but "there are 47 distinct cells, each with its own boundary."

## Why this matters for the interview

Sartorius makes microscopes, cell counters, and bioprocessing instruments. Every one of those needs to answer the question "how many cells, where, and how big?" Instance segmentation is the model class that answers it. When the job description says "AI features intuitive in customer workflows," this is one of those features.

## 2. How PyTorch represents images

Everything in PyTorch is a **tensor** — a multidimensional array with a dtype and a device.

An image is a tensor with shape `(C, H, W)` or `(H, W, C)` depending on convention:

- `C` = number of channels (3 for RGB, 1 for grayscale)
- `H` = height in pixels
- `W` = width in pixels

PyTorch typically uses `(C, H, W)`. PIL and most image libraries use `(H, W, C)`. You will spend half your debugging time on dimension mismatches. Get used to printing `.shape`.

In [ ]:
import torch
import numpy as np

# Check CUDA availability. On your RTX 3060 this should print True.
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Make a fake grayscale image (a tensor of random values) so you can poke at shapes.
fake_image = torch.rand(1, 256, 256)  # (channels, height, width)
print(f"shape: {fake_image.shape}")
print(f"dtype: {fake_image.dtype}")
print(f"device: {fake_image.device}")
print(f"min/max: {fake_image.min():.3f} / {fake_image.max():.3f}")

# Move it to GPU.
if torch.cuda.is_available():
    fake_image_gpu = fake_image.cuda()
    print(f"After .cuda(): {fake_image_gpu.device}")

## 3. What Cellpose is

**One paragraph version.** Cellpose is a deep learning model for segmenting cells in microscopy images. Under the hood it is a **U-Net** — an encoder-decoder convolutional neural network where the encoder downsamples the image into a compact feature representation and the decoder upsamples it back to the original resolution, producing a per-pixel output. Cellpose's special trick is that instead of predicting a binary mask directly, it predicts a **flow field** — for every pixel, the direction toward the center of the cell it belongs to. Then a post-processing step follows those flows to group pixels into distinct cell instances. This trick is what makes it good at separating touching cells.

**What you should be able to say in an interview:**

- "Cellpose is a U-Net trained on biological imagery."
- "It predicts a flow field per pixel and groups pixels by following flows to attractor points."
- "I used the `cyto3` pretrained model because it generalizes well across cell types."
- "For better Sartorius-specific performance I would fine-tune on the Kaggle competition data."

## What you do NOT need to be able to say

- The exact math of the flow field
- The skip-connection details of U-Net
- The pixel-clustering algorithm

Those are PhD-thesis-grade. Knowing them is great. Not knowing them is normal. Bluffing them is bad.

## 4. Load the model and run it on a real image

The first time you run this it will download the model weights (~25 MB). After that the load is from disk and takes a few seconds.

In [ ]:
from cellpose import models

# gpu=True will use CUDA if available. model_type='cyto3' is the latest general
# cytoplasm model. Other options: 'cyto2', 'nuclei', or a path to custom weights.
model = models.Cellpose(gpu=True, model_type='cyto3')
print("Model loaded.")

In [ ]:
from pathlib import Path
from PIL import Image

# Cellpose ships a few example images. We use one as a quick sanity check.
# If you have your own microscopy image, drop it in ../data/sample_images/ and
# change this path.
sample_paths = list(Path('../data/sample_images').glob('*'))
if not sample_paths:
    # Fall back to a Cellpose built-in example.
    import cellpose, os
    sample_paths = [Path(os.path.dirname(cellpose.__file__)) / 'data' / 'rgb_2D_tif.tif']

sample_path = sample_paths[0]
print(f"Using image: {sample_path}")

img = Image.open(sample_path).convert('RGB')
arr = np.array(img)
print(f"Image shape: {arr.shape} (height, width, channels)")
print(f"Image dtype: {arr.dtype}")
print(f"Pixel value range: {arr.min()} to {arr.max()}")

In [ ]:
import time

# Run inference. Cellpose takes a LIST of images and returns LISTS.
# channels=[0, 0] = use a single grayscale channel (no nuclei stain).
# diameter=None = auto-estimate cell size.
t0 = time.perf_counter()
masks, flows, styles, diams = model.eval([arr], channels=[0, 0], diameter=None)
elapsed = (time.perf_counter() - t0) * 1000

mask = masks[0]
print(f"Inference: {elapsed:.0f} ms")
print(f"Mask shape: {mask.shape}")
print(f"Mask dtype: {mask.dtype}")
print(f"Number of cells detected: {mask.max()}")
print(f"Estimated cell diameter: {diams[0]:.1f} pixels")

## 5. Understanding the mask output

The mask is a 2D array the same height and width as the input image, where each pixel value is the **integer ID** of the cell it belongs to:

- `0` = background
- `1` = cell #1
- `2` = cell #2
- ... and so on up to `mask.max()`

This is the instance-segmentation output. To make a binary cell-vs-background mask you would just write `mask > 0`.

In [ ]:
# How many pixels belong to each cell? (size = area in pixels)
from collections import Counter

pixel_counts = Counter(mask.flatten().tolist())
del pixel_counts[0]  # drop background

sizes = sorted(pixel_counts.values(), reverse=True)
print(f"Largest cell: {sizes[0]} pixels")
print(f"Smallest cell: {sizes[-1]} pixels")
print(f"Median cell size: {sizes[len(sizes)//2]} pixels")

## 6. Visualize

A picture is worth more than a hundred print statements when you are debugging segmentation.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(arr)
axes[0].set_title('Original')
axes[0].axis('off')

# 'nipy_spectral' colormap gives each cell ID a distinct color.
axes[1].imshow(arr)
axes[1].imshow(np.ma.masked_where(mask == 0, mask), alpha=0.5, cmap='nipy_spectral')
axes[1].set_title(f'Segmented ({int(mask.max())} cells)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 7. Latency — CPU vs GPU

**This is the deployment-engineering money cell.** The job posting explicitly mentions "optimize models for latency and hardware constraints." Run this and remember the numbers. You will quote them in the interview.

The pattern: load the same model twice, once on CPU and once on GPU, run inference on the same image, and compare wall-clock time.

In [ ]:
# WARM-UP RUN — the first inference is always slower because of CUDA kernel
# compilation and memory allocation. We discard it from measurements.
_ = model.eval([arr], channels=[0, 0], diameter=diams[0])

# Measure GPU
N_TRIALS = 5
gpu_times = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    model.eval([arr], channels=[0, 0], diameter=diams[0])
    gpu_times.append((time.perf_counter() - t0) * 1000)

print(f"GPU: mean {np.mean(gpu_times):.1f} ms, std {np.std(gpu_times):.1f} ms")
print(f"  individual runs: {[f'{t:.0f}' for t in gpu_times]}")

In [ ]:
# Now CPU. This will be slow. Note that 'cyto3' is a small model — for larger
# models the gap is even more dramatic.
model_cpu = models.Cellpose(gpu=False, model_type='cyto3')
_ = model_cpu.eval([arr], channels=[0, 0], diameter=diams[0])  # warm-up

cpu_times = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    model_cpu.eval([arr], channels=[0, 0], diameter=diams[0])
    cpu_times.append((time.perf_counter() - t0) * 1000)

print(f"CPU: mean {np.mean(cpu_times):.1f} ms, std {np.std(cpu_times):.1f} ms")
print(f"  individual runs: {[f'{t:.0f}' for t in cpu_times]}")
print(f"\nSpeedup: {np.mean(cpu_times) / np.mean(gpu_times):.1f}x")

## 8. Per-cell morphology

Once you have an instance mask, downstream measurements are mostly bookkeeping. This is the kind of "real-time insights in customer workflows" the job description talks about.

In [ ]:
from skimage.measure import regionprops_table
import pandas as pd

# Properties to extract per cell. Each is a research-grade measurement.
props = regionprops_table(
    mask,
    properties=['label', 'area', 'perimeter', 'eccentricity', 'centroid'],
)
df = pd.DataFrame(props)
df.columns = ['cell_id', 'area_px', 'perimeter_px', 'eccentricity', 'centroid_y', 'centroid_x']
df.head(10)

In [ ]:
# Quick summary stats — the kind of thing an instrument would surface to the user.
print(f"Cells detected: {len(df)}")
print(f"Mean area: {df['area_px'].mean():.0f} px")
print(f"Mean eccentricity: {df['eccentricity'].mean():.3f}")
print(f"  (0 = perfect circle, 1 = highly elongated)")

## 9. From notebook to service

Everything you just did in this notebook is now the **inference path** of the FastAPI service in `backend/inference.py` and `backend/main.py`. Read those two files alongside this notebook. The 1:1 mapping is:

| Notebook step | Service code |
|---|---|
| `models.Cellpose(gpu=True, ...)` | `CellSegmenter.__init__` |
| `model.eval([arr], ...)` | `CellSegmenter.segment` |
| Image.open(...) on a file | `await file.read()` in the `/segment` route |
| Latency measurement here | `LATENCY_WINDOW` deque + `/metrics` |
| Plotting in matplotlib | Base64-encoded PNG sent to the browser |

**The conceptual jump:** in a notebook the model lives in your kernel and you call it directly. In a service the model lives in a long-running process and the world calls it over HTTP. That's the only difference. Everything else is plumbing.

## What to learn next (post-Loopback)

1. **Docker** — wrap the backend in a container so it can run anywhere CUDA is available.
2. **Fine-tuning** — train a small U-Net on the actual Sartorius Kaggle data instead of using `cyto3` pretrained. You will learn dataset construction, training loops, and validation.
3. **ONNX export** — convert the trained model to ONNX, the format Sartorius would actually deploy. This unlocks model-level latency optimizations.
4. **Batching** — add a `/segment_batch` endpoint that processes multiple images per call. Discuss throughput vs latency tradeoff.

Each of those is a real interview talking point. Build them in that order.